# EDEN ISS Space Greenhouse: Exploratory Biomass & Yield Analysis
### NASA OSDR Study OSD-895 / LSDS-160 | Frontiers in Plant Science (PMC7264257)

This notebook explores the biomass production, growth cycles, and environmental metrics from the **EDEN ISS Mobile Test Facility (MTF)** operated near Neumayer Station III in Antarctica during the 2018 experiment campaign (286 days).

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)

# Load processed FAIR data
crops_df = pd.read_csv("../data/processed/eden_iss_crops.csv")
nutrients_df = pd.read_csv("../data/processed/nutrient_solutions.csv")
with open("../data/processed/greenhouse_summary.json") as f:
    summary = json.load(f)

print(f"Loaded {len(crops_df)} crop evaluation cohorts.")
print(f"Total Fresh Harvest: {summary['total_fresh_edible_biomass_kg']} kg over {summary['duration_days']} days.")
crops_df.head()

## 1. Crop Biomass & Daily Productivity Overview
In space life-support systems (BLSS/ECLSS), area and time are critical constraints. The primary metric is **Daily Edible Productivity** ($g/m^2/day$).

In [ ]:
# Sort by daily productivity
sorted_df = crops_df.sort_values("daily_edible_productivity_g_m2_day", ascending=False)

plt.figure(figsize=(12, 7))
palette = {
    "Fruit Crops": "#f43f5e",
    "Leafy Greens": "#10b981",
    "Lettuce": "#059669",
    "Herbs": "#0ea5e9",
    "Tuber Crops": "#f59e0b"
}

barplot = sns.barplot(
    data=sorted_df,
    x="daily_edible_productivity_g_m2_day",
    y="display_name",
    hue="crop_category",
    dodge=False,
    palette=palette
)

plt.title("EDEN ISS Daily Edible Biomass Productivity by Crop (g/m²/day)", fontsize=14, weight="bold")
plt.xlabel("Daily Productivity (g / m² / day)")
plt.ylabel("Crop & Cultivar")
plt.legend(title="Crop Category", loc="lower right")
plt.tight_layout()
plt.show()

## 2. Growth Duration vs. Yield
How does cultivation duration compare with fresh yield produced per square meter?

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=crops_df,
    x="growth_time_days",
    y="mean_edible_yield_kg_m2",
    hue="crop_category",
    size="daily_edible_productivity_g_m2_day",
    sizes=(60, 400),
    palette=palette,
    alpha=0.85
)

for _, row in crops_df.iterrows():
    plt.text(row["growth_time_days"] + 4, row["mean_edible_yield_kg_m2"], row["cultivar"], fontsize=9, alpha=0.8)

plt.title("Cultivation Cycle Duration vs. Fresh Edible Yield", fontsize=14, weight="bold")
plt.xlabel("Cultivation Cycle (Days)")
plt.ylabel("Mean Fresh Edible Yield (kg/m²)")
plt.tight_layout()
plt.show()

## 3. Light Intensity Treatment Experiment (Arugula)
Arugula was cultivated under standard light ($330\ \mu mol/m^2/s$) and high light ($600\ \mu mol/m^2/s$).

In [ ]:
arugula_df = crops_df[crops_df["sample_id"].str.contains("Arugula")]
print(arugula_df[["display_name", "light_intensity_umol_m2_s", "growth_time_days", "mean_edible_yield_kg_m2", "daily_edible_productivity_g_m2_day"]])

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
sns.barplot(data=arugula_df, x="light_intensity_umol_m2_s", y="daily_edible_productivity_g_m2_day", ax=ax[0], palette="Blues_d")
ax[0].set_title("Daily Productivity (g/m²/d)")
ax[0].set_xlabel("Light Intensity (µmol/m²/s)")

sns.barplot(data=arugula_df, x="light_intensity_umol_m2_s", y="mean_edible_yield_kg_m2", ax=ax[1], palette="Greens_d")
ax[1].set_title("Total Fresh Yield (kg/m²)")
ax[1].set_xlabel("Light Intensity (µmol/m²/s)")

plt.suptitle("Arugula Light Treatment Response", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 4. Life-Support Mission Diet Sizing Simulation
Based on actual Antarctic greenhouse productivity, what active growing area is required for human space exploration?

In [ ]:
def calculate_mission_footprint(crew_size=6, daily_salad_grams_per_person=100, mission_days=500, productivity_rate=75.0):
    daily_crew_grams = crew_size * daily_salad_grams_per_person
    growing_area_m2 = daily_crew_grams / productivity_rate
    total_biomass_kg = (daily_crew_grams * mission_days) / 1000.0
    return {
        "crew_size": crew_size,
        "daily_ration_g": daily_salad_grams_per_person,
        "mission_days": mission_days,
        "growing_area_m2": round(growing_area_m2, 2),
        "total_biomass_kg": round(total_biomass_kg, 1),
        "eden_greenhouse_ratio": round(growing_area_m2 / 12.5, 2)
    }

scenario_mars = calculate_mission_footprint(crew_size=4, daily_salad_grams_per_person=100, mission_days=680)
scenario_moon = calculate_mission_footprint(crew_size=6, daily_salad_grams_per_person=150, mission_days=180)

print("Mars Transit/Surface Scenario (4 crew, 100g/d):", scenario_mars)
print("Lunar Base Scenario (6 crew, 150g/d):", scenario_moon)